# The Profile — VertiPaq Analyzer

What is taking up space in this semantic model?

We will compare **Trudoso10K** and **Trudoso10K_WithAutoDateXL**, then inspect the hidden date tables and the columns that consume the most memory.

The analyzer gives us evidence: model size, table size, column cardinality, dictionary size, and relationship statistics. A large object is a starting point for investigation, not proof of a modeling mistake.

## Setup

Run the shared setup notebook, using the same process as the other demos. Attach **EvidenceLocker** as the default Lakehouse first, as required by setup.

Both Trudoso models must already be published and processed in this workspace. The inspection cells read their statistics; they do not refresh or modify either model.

In [ ]:
%run 00_Setup_Fresh

In [ ]:
import pandas as pd

DATASET = "Trudoso10K"
DATASETXL = "Trudoso10K_WithAutoDateXL"
WORKSPACE = notebookutils.runtime.context["currentWorkspaceId"]

## 1. The baseline

Open VertiPaq Analyzer for **Trudoso10K**. Start with the model size, then inspect its tables.

The function displays an interactive analyzer and also returns numeric DataFrames for further analysis.

In [ ]:
print(f"=== {DATASET} ===")
vpa_base = labs.vertipaq_analyzer(
    dataset=DATASET, workspace=WORKSPACE, dark_mode=True
)

## 2. The auto date/time model

Now open **Trudoso10K_WithAutoDateXL**. Compare its size with the baseline and look for hidden tables whose names begin with `LocalDateTable`.

Auto date/time creates hidden date tables for eligible date columns. Each covers full calendar years spanning the column's dates. Those tables consume memory and take time to refresh.

In [ ]:
print(f"=== {DATASETXL} ===")
vpa_xl = labs.vertipaq_analyzer(
    dataset=DATASETXL, workspace=WORKSPACE, dark_mode=True
)

## 3. Follow the memory

Compare the measured sizes, then inspect the auto-generated date tables. The numbers below come from these models; they are not a general multiplier for auto date/time.

In [ ]:
comparison = pd.concat(
    [vpa_base["Model Summary"], vpa_xl["Model Summary"]],
    ignore_index=True,
)[["Dataset Name", "Total Size", "Table Count", "Column Count"]].copy()
comparison["Size MiB"] = (comparison["Total Size"] / (1024 ** 2)).round(2)
display(comparison)

xl_tables = vpa_xl["Tables"]
date_tables = xl_tables[
    xl_tables["Table Name"].str.startswith(
        ("LocalDateTable_", "DateTableTemplate_"), na=False
    )
]
display(
    date_tables[["Table Name", "Row Count", "Total Size", "% DB"]]
    .sort_values("Total Size", ascending=False)
)
print(f"Auto-generated date-table size: {date_tables['Total Size'].sum() / (1024 ** 2):,.2f} MiB")

## Optional: investigate individual tables and columns

For a short bonus demo, the comparison above is a natural stopping point.

To continue, use the XL results already collected. Rank tables by **Total Size**, then compare column size, cardinality, and dictionary size. Bytes and row counts answer different questions.

In [ ]:
# Reuse the existing scan rather than running the analyzer again.
vpa = vpa_xl
df_tables = vpa["Tables"]
df_columns = vpa["Columns"]

display(
    df_tables[["Table Name", "Row Count", "Total Size", "Dictionary Size", "% DB"]]
    .sort_values("Total Size", ascending=False)
    .head(15)
)

In [ ]:
# Which columns account for the most memory?
column_details = [
    "Table Name", "Column Name", "Cardinality", "Data Type",
    "Encoding", "Total Size", "Data Size", "Dictionary Size", "% DB",
]
display(
    df_columns[column_details]
    .sort_values("Total Size", ascending=False)
    .head(15)
)

# Which columns have the most distinct values?
display(
    df_columns[column_details]
    .sort_values("Cardinality", ascending=False)
    .head(15)
)

### Interpret the evidence

- **Cardinality:** many distinct values can increase storage requirements, especially for text dictionaries. Inspect the measured size before deciding what matters.
- **Dictionary size:** consider whether a large text column is needed, whether its granularity is appropriate, and whether a smaller representation preserves the business meaning.
- **Encoding:** VALUE and HASH are different compression approaches; VALUE is not automatically worse. In SLL 0.18.1, the returned `Encoding` field reflects the column's encoding hint. Treat it as metadata rather than proof of the engine's actual storage choice.
- **Hidden date tables:** review whether shared, intentional date dimensions would better serve the model's reporting needs.

A smaller model does not automatically mean every query is faster. Validate the effect of a change against the queries and reports that matter.

In [ ]:
# Relationship statistics provide another line of evidence.
display(vpa["Relationships"])

## Optional: preserve the results as CSV files

These files preserve the numeric DataFrames for comparison or sharing. They are ordinary CSV files, not DAX Studio VPAX files.

SLL 0.18.1 does not accept `export="zip"` in `vertipaq_analyzer()`. The optional cell below saves the results we already collected instead of running another scan.

In [ ]:
SAVE_CSV = False  # Set True when you want to save this snapshot.

if SAVE_CSV:
    from datetime import datetime, timezone
    from pathlib import Path

    folder = Path("/lakehouse/default/Files/VertiPaq") / datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%S%fZ")
    folder.mkdir(parents=True, exist_ok=False)

    for model_name, results in [(DATASET, vpa_base), (DATASETXL, vpa_xl)]:
        for section, frame in results.items():
            filename = f"{model_name}_{section.replace(' ', '_')}.csv"
            frame.to_csv(folder / filename, index=False)
    print(f"Saved CSV snapshots to {folder}")
else:
    print("CSV export skipped.")

## Optional: collect statistics over time

Exporting with `export="table"` appends statistics and run metadata to Lakehouse Delta tables. It performs a new scan of the selected model.

Use this when you want to compare sizes across releases. It is separate from the read-only inspection above and writes to the attached Lakehouse.

In [ ]:
EXPORT_TABLES = False  # Set True to scan and append a Lakehouse snapshot.

if EXPORT_TABLES:
    labs.vertipaq_analyzer(
        dataset=DATASETXL,
        workspace=WORKSPACE,
        export="table",
    )
else:
    print("Delta-table export skipped.")

## What to take away

Start with **where the bytes are**, identify the tables and columns responsible, and investigate why they are there.

| Question | Evidence |
|---|---|
| How large is the model? | Model Summary: Total Size |
| Which tables dominate? | Tables ranked by Total Size |
| What do the hidden date tables cost? | Their row counts and Total Size |
| Which columns deserve attention? | Total Size, Cardinality, Dictionary Size |
| How does the model change over time? | Saved CSV snapshots or Delta-table exports |

### Resources
- [Semantic Link Labs](https://github.com/microsoft/semantic-link-labs)
- [Auto date/time in Power BI](https://learn.microsoft.com/en-us/power-bi/transform-model/desktop-auto-date-time)
- [Auto date/time guidance](https://learn.microsoft.com/en-us/power-bi/guidance/auto-date-time)
- [DAX Studio](https://daxstudio.org)
- [Data Shepherd](https://datashepherd.dev)